In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, os, pickle, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import (SPE1_PICKLE_ROOT, DICT_CELL_TYPE, DICT_PATCH_TYPE,
                    DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV)
from pop_ridge_utils import load_population_results
from ridge_regression_utils import WAVEFORM_LABELS, FEAT_LABELS

warnings.filterwarnings('ignore')

_CB = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#56B4E9', '#E69F00']
_SIG_COL   = '#D55E00'
_INSIG_COL = '#56B4E9'

In [ ]:
# ── Load ridge regression results ─────────────────────────────────────────────
RIDGE_PICKLE_DIR = os.path.join(SPE1_PICKLE_ROOT, 'ridge_regression_pickles')

all_results, cell_ids, target_names, predictor_sets = load_population_results(RIDGE_PICKLE_DIR)

feat_labels   = FEAT_LABELS
target_labels = (
    [f'Pre {l}'     for l in feat_labels] +
    [f'Pre−BL {l}'  for l in feat_labels] +
    [f'Post {l}'    for l in feat_labels] +
    [f'Post−BL {l}' for l in feat_labels] +
    [f'Δ {l}'       for l in feat_labels]
)

# Load population summary pickle
pop_path = os.path.join(RIDGE_PICKLE_DIR, 'population_ridge_results.pkl')
with open(pop_path, 'rb') as f:
    pop = pickle.load(f)

r2_pop        = pop['r2_pop']
sig_pop       = pop['sig_pop']
beta_pop      = pop['beta_pop']
df_tests      = pop['df_tests']
mean_beta_mat = pop['mean_beta_mat']
sig_beta_mat  = pop['sig_beta_mat']

print(f'{len(cell_ids)} cells  |  {len(target_names)} targets')

In [ ]:
# ── Build per-cell metadata + ridge stats DataFrame ───────────────────────────
rows = []
for cid in cell_ids:
    cell_num = int(cid.replace('c', ''))
    res = all_results[cid]

    row = dict(
        cell_id      = cid,
        cell_num     = cell_num,
        cell_type    = DICT_CELL_TYPE.get(cell_num, 'unknown'),
        patch_type   = DICT_PATCH_TYPE.get(cell_num, 'unknown'),
        cort_depth   = DICT_CORT_DEPTH.get(cell_num, np.nan),
        dark_neuron  = str(DICT_DARK_NEURONS.get(cell_num, np.nan)),
        eap_visible  = str(DICT_EAP_WAV.get(cell_num, np.nan)),
        n_spikes     = res[target_names[0]]['Waveform only'].get('n_valid', np.nan),
        n_sig_wv     = sum(res[tn]['Waveform only'].get('sig_fdr', False) for tn in target_names),
        n_sig_isi    = sum(res[tn]['Log ISI only'].get('sig_fdr', False) for tn in target_names),
        n_sig_both   = sum(res[tn]['Waveform + Log ISI'].get('sig_fdr', False) for tn in target_names),
        mean_r2_wv   = float(np.nanmean([res[tn]['Waveform only'].get('r2_cv', np.nan) for tn in target_names])),
    )
    for tn, tl in zip(target_names, target_labels):
        row[f'r2_{tn}']    = res[tn]['Waveform only'].get('r2_cv', np.nan)
        row[f'sig_{tn}']   = res[tn]['Waveform only'].get('sig_fdr', False)
        row[f'alpha_{tn}'] = res[tn]['Waveform only'].get('best_alpha', np.nan)
    rows.append(row)

df = pd.DataFrame(rows)

# Replace 'nan' strings with actual NaN so they're excluded from plots
for col in ['dark_neuron', 'eap_visible']:
    df[col] = df[col].replace('nan', np.nan)

print(df[['cell_id','cell_type','dark_neuron','eap_visible','n_spikes','n_sig_wv','mean_r2_wv']].to_string(index=False))

In [ ]:
# ── Cell × target significance heatmap ────────────────────────────────────────
sig_matrix = np.array(
    [[bool(df.loc[df.cell_id == cid, f'sig_{tn}'].values[0])
      for tn in target_names]
     for cid in cell_ids],
    dtype=float
)

n_sig_per_cell  = sig_matrix.sum(axis=1)
sort_idx        = np.argsort(n_sig_per_cell)[::-1]
cell_ids_sorted = [cell_ids[i] for i in sort_idx]
n_cells         = len(cell_ids)

row_h = 0.45   # inches per cell row
fig_h = max(8, row_h * n_cells + 3)
fig, ax = plt.subplots(figsize=(14, fig_h))

im = ax.imshow(sig_matrix[sort_idx], aspect='auto', cmap='Blues', vmin=0, vmax=1,
               extent=[-0.5, len(target_names) - 0.5, n_cells - 0.5, -0.5])

for d in [4.5, 9.5]:
    ax.axvline(d, color='white', lw=2.5)
for x, lbl in [(2, 'Pre (abs)'), (7, 'Pre − BL'), (12, 'Δ post−pre')]:
    ax.text(x, n_cells + 0.8, lbl, ha='center', va='top', fontsize=10,
            fontweight='bold', color='#444')

ax.set_xticks(range(len(target_names)))
ax.set_xticklabels([tl.split(' ', 1)[-1] for tl in target_labels],
                   rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(n_cells))
ax.set_yticklabels(
    [f'{cell_ids_sorted[i]}  ({int(n_sig_per_cell[sort_idx[i]])})'
     for i in range(n_cells)],
    fontsize=8.5
)
ax.set_xlabel('LFP target', fontsize=11)
ax.set_ylabel('Cell  (n significant targets)', fontsize=11)
ax.set_title('Which cells have FDR-significant Waveform-only models?\n'
             '(sorted by total n significant)', fontsize=12, pad=16)

cbar = plt.colorbar(im, ax=ax, shrink=0.35, pad=0.01)
cbar.set_ticks([0, 1]); cbar.set_ticklabels(['Not sig.', 'Sig.'])

# Fraction per target column (above the matrix)
for c in range(len(target_names)):
    frac = sig_matrix[:, c].mean()
    ax.text(c, -0.7, f'{frac:.0%}', ha='center', va='bottom',
            fontsize=7.5, color='#555')

fig.tight_layout()
plt.show()

print(f'\nCells with ≥1 significant target: {(n_sig_per_cell > 0).sum()}/{n_cells}')
print(f'Targets with ≥1 significant cell: {(sig_matrix.sum(axis=0) > 0).sum()}/{len(target_names)}')

In [ ]:
# ── CV R² distribution per target, grouped by target type ─────────────────────
group_colors = {'Pre (abs)': '#1976D2', 'Pre − BL': '#43A047', 'Δ post−pre': '#E53935'}
group_map = (
    [(tn, 'Pre (abs)')  for tn in target_names[:5]]  +
    [(tn, 'Pre − BL')   for tn in target_names[5:10]] +
    [(tn, 'Δ post−pre') for tn in target_names[10:]]
)
feat_short = ['Amp', 'Std', 'Gamma', 'Exp', 'Theta']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('CV R² across cells — Waveform only model', fontsize=13, y=1.01)

for ax, (grp_label, col) in zip(axes, [
    ('Pre (abs)',  '#1976D2'),
    ('Pre − BL',  '#43A047'),
    ('Δ post−pre','#E53935'),
]):
    grp_targets = [(tn, fs) for (tn, grp), fs in zip(group_map, feat_short * 3) if grp == grp_label]
    rng = np.random.default_rng(42)

    for pos, (tn, fs) in enumerate(grp_targets):
        vals = df[f'r2_{tn}'].dropna().values
        bp = ax.boxplot(vals, positions=[pos], widths=0.55, patch_artist=True,
                        medianprops=dict(color='k', lw=2.5),
                        boxprops=dict(facecolor=col, alpha=0.4),
                        whiskerprops=dict(color='#555', lw=1.2),
                        capprops=dict(color='#555', lw=1.2),
                        flierprops=dict(marker='o', markersize=3, alpha=0.3, color=col))
        jitter = rng.uniform(-0.2, 0.2, len(vals))
        ax.scatter(pos + jitter, vals, alpha=0.6, s=18, color=col, zorder=4)
        # Annotate median
        med = np.median(vals)
        ax.text(pos, ax.get_ylim()[0] if ax.get_ylim()[0] > -0.1 else -0.05,
                f'{med:+.3f}', ha='center', va='top', fontsize=7, color='#333')

    ax.axhline(0, color='gray', lw=1.2, ls='--', alpha=0.7)
    ax.set_xticks(range(len(grp_targets)))
    ax.set_xticklabels([fs for _, fs in grp_targets], fontsize=10)
    ax.set_title(grp_label, fontsize=11, fontweight='bold', color=col, pad=8)
    ax.set_xlabel('LFP feature', fontsize=9)
    if ax == axes[0]:
        ax.set_ylabel('5-fold CV R²', fontsize=10)
    sns.despine(ax=ax)

fig.tight_layout()
plt.show()

In [ ]:
# ── Best alpha distribution per target (log scale), grouped ───────────────────
feat_short = ['Amp', 'Std', 'Gamma', 'Exp', 'Theta']
grp_info   = [('Pre (abs)', '#1976D2', target_names[:5]),
              ('Pre − BL',  '#43A047', target_names[5:10]),
              ('Δ post−pre','#E53935', target_names[10:])]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
fig.suptitle('Selected Ridge alpha per target (Waveform only)\n'
             'High alpha = strong regularization needed (little signal)', fontsize=12)

for ax, (grp_lbl, col, tns) in zip(axes, grp_info):
    for pos, (tn, fs) in enumerate(zip(tns, feat_short)):
        vals = np.log10(df[f'alpha_{tn}'].dropna().values.clip(1e-4))
        ax.boxplot(vals, positions=[pos], widths=0.55, patch_artist=True,
                   medianprops=dict(color='k', lw=2.5),
                   boxprops=dict(facecolor=col, alpha=0.4),
                   whiskerprops=dict(color='#555', lw=1.2),
                   capprops=dict(color='#555', lw=1.2),
                   flierprops=dict(marker='o', markersize=3, alpha=0.3, color=col))

    ax.set_xticks(range(5))
    ax.set_xticklabels(feat_short, fontsize=10)
    ax.set_title(grp_lbl, fontsize=11, fontweight='bold', color=col, pad=8)
    ax.set_xlabel('LFP feature', fontsize=9)
    if ax == axes[0]:
        ax.set_ylabel('log₁₀(best alpha)', fontsize=10)
    # Reference line at alpha=1
    ax.axhline(0, color='gray', lw=1, ls='--', alpha=0.5, label='α=1')
    sns.despine(ax=ax)

axes[0].legend(fontsize=8, frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# ── Does metadata predict mean CV R²? ─────────────────────────────────────────
cat_vars = {
    'Cell type':    'cell_type',
    'Patch type':   'patch_type',
    'Dark neuron':  'dark_neuron',
    'EAP visible':  'eap_visible',
}

fig, axes = plt.subplots(1, len(cat_vars), figsize=(4.5 * len(cat_vars), 5))
fig.suptitle('Does cell metadata predict mean CV R²? (Waveform only)', fontsize=13)

for ax, (label, col) in zip(axes, cat_vars.items()):
    d = df[['mean_r2_wv', col]].dropna()
    order = sorted(d[col].unique())
    if len(order) < 2:
        ax.set_visible(False); continue
    palette = dict(zip(order, _CB[:len(order)]))

    sns.boxplot(data=d, x=col, y='mean_r2_wv', order=order, palette=palette,
                width=0.5, ax=ax, fliersize=0, boxprops={'alpha': 0.45},
                medianprops=dict(color='k', lw=2))
    sns.stripplot(data=d, x=col, y='mean_r2_wv', order=order, palette=palette,
                  alpha=0.75, jitter=0.18, size=7, ax=ax)
    ax.axhline(0, color='gray', lw=1, ls='--', alpha=0.6)

    # Group means
    for i, g in enumerate(order):
        m = d[d[col] == g]['mean_r2_wv'].mean()
        ax.hlines(m, i - 0.3, i + 0.3, color='k', lw=2, zorder=5)

    # Stat test
    groups = [d[d[col] == g]['mean_r2_wv'].values for g in order]
    valid_groups = [g for g in groups if len(g) >= 3]
    if len(valid_groups) >= 2:
        _, p = stats.kruskal(*valid_groups)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        color = _SIG_COL if sig != 'ns' else '#888'
        y_top = d['mean_r2_wv'].max()
        y_rng = ax.get_ylim()[1] - ax.get_ylim()[0]
        ax.text(0.5, 0.97, sig, transform=ax.transAxes, ha='center', va='top',
                fontsize=14, fontweight='bold', color=color)
        ax.text(0.5, 0.91, f'p={p:.3f}', transform=ax.transAxes,
                ha='center', va='top', fontsize=8, color='#555')

    # Annotate n per group
    for i, (g, grp) in enumerate(zip(order, groups)):
        ax.text(i, ax.get_ylim()[0], f'n={len(grp)}', ha='center', va='bottom',
                fontsize=8, color='#555')

    ax.set_title(label, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('')
    ax.set_xticklabels(order, fontsize=9)
    if ax == axes[0]:
        ax.set_ylabel('Mean CV R² across targets', fontsize=10)
    else:
        ax.set_ylabel('')
    sns.despine(ax=ax)

fig.tight_layout()
plt.show()

# ── Spearman: cortical depth ──
d_depth = df[['mean_r2_wv', 'cort_depth']].dropna()
if len(d_depth) >= 5:
    rho, p = stats.spearmanr(d_depth['cort_depth'], d_depth['mean_r2_wv'])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(d_depth['cort_depth'], d_depth['mean_r2_wv'],
               color=_CB[0], s=60, alpha=0.8, edgecolors='white', lw=0.5)
    # Trend line
    m, b = np.polyfit(d_depth['cort_depth'], d_depth['mean_r2_wv'], 1)
    xs = np.linspace(d_depth['cort_depth'].min(), d_depth['cort_depth'].max(), 100)
    color = _SIG_COL if p < 0.05 else '#888'
    ax.plot(xs, m * xs + b, color=color, lw=2, ls='--' if p >= 0.05 else '-')
    ax.set_xlabel('Cortical depth (µm)', fontsize=10)
    ax.set_ylabel('Mean CV R²', fontsize=10)
    ax.set_title(f'Cortical depth × mean R²\nSpearman ρ = {rho:.3f},  p = {p:.3f}', fontsize=11)
    sns.despine(ax=ax)
    plt.tight_layout(); plt.show()
else:
    print('Not enough cells with depth data.')

In [ ]:
# ── Sample size confound check ─────────────────────────────────────────────────
rho, p = stats.spearmanr(df['n_spikes'].dropna(), df['mean_r2_wv'].dropna())

fig, ax = plt.subplots(figsize=(5, 4))
sc = ax.scatter(df['n_spikes'], df['mean_r2_wv'],
                c=df['n_sig_wv'], cmap='YlOrRd', s=60,
                alpha=0.85, edgecolors='white', lw=0.5, vmin=0)
plt.colorbar(sc, ax=ax, label='N sig. targets', shrink=0.7)
ax.axhline(0, color='gray', lw=1, ls='--', alpha=0.6)

m, b = np.polyfit(df['n_spikes'].dropna(), df['mean_r2_wv'].dropna(), 1)
xs = np.linspace(df['n_spikes'].min(), df['n_spikes'].max(), 100)
line_col = _SIG_COL if p < 0.05 else '#888'
ax.plot(xs, m * xs + b, color=line_col, lw=2, ls='-' if p < 0.05 else '--')

ax.set_xlabel('N spikes', fontsize=10)
ax.set_ylabel('Mean CV R² (Waveform only)', fontsize=10)
ax.set_title(f'Sample size vs model performance\nSpearman ρ = {rho:.3f},  p = {p:.3f}',
             fontsize=11)
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
ax.text(0.97, 0.05, sig, transform=ax.transAxes, ha='right', va='bottom',
        fontsize=14, fontweight='bold', color=line_col)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

if p < 0.05:
    print('⚠ Sample size significantly predicts R² — consider this a potential confound.')
else:
    print('✓ Sample size does not predict R² — performance reflects real signal, not n.')

In [ ]:
# ── Beta direction × metadata for FDR-significant population pairs ─────────────
sig_pairs = df_tests[df_tests.sig_fdr][['target','target_label','feature']].values.tolist()

if not sig_pairs:
    print('No population-level FDR-significant pairs.')
else:
    for tn, tl, feat in sig_pairs:
        beta_vals = np.array([beta_pop[tn][feat][i] for i, cid in enumerate(cell_ids)])
        df_b = df[['cell_id','cell_type','patch_type','dark_neuron','eap_visible']].copy()
        df_b['beta'] = beta_vals
        df_b = df_b.dropna(subset=['beta'])

        ncols = len(cat_vars)
        fig, axes = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4.5), sharey=True)
        fig.suptitle(f'Beta direction by metadata\n{tl}  ×  {feat}', fontsize=11)

        for ax, (label, col) in zip(axes, cat_vars.items()):
            d = df_b[['beta', col]].dropna()
            order = sorted(d[col].unique())
            if len(order) < 2:
                ax.set_visible(False); continue
            palette = dict(zip(order, _CB[:len(order)]))

            sns.boxplot(data=d, x=col, y='beta', order=order, palette=palette,
                        width=0.5, ax=ax, fliersize=0, boxprops={'alpha': 0.4},
                        medianprops=dict(color='k', lw=2))
            sns.stripplot(data=d, x=col, y='beta', order=order, palette=palette,
                          alpha=0.75, jitter=0.18, size=7, ax=ax)
            ax.axhline(0, color='gray', lw=1.2, ls='--', alpha=0.7)

            # Mean line
            for i, g in enumerate(order):
                m = d[d[col] == g]['beta'].mean()
                ax.hlines(m, i - 0.28, i + 0.28, color='k', lw=2.2, zorder=5)

            # Stat + n
            groups = [d[d[col] == g]['beta'].values for g in order]
            valid_groups = [g for g in groups if len(g) >= 3]
            if len(valid_groups) >= 2:
                _, p_kw = stats.kruskal(*valid_groups)
                sig = '***' if p_kw < 0.001 else '**' if p_kw < 0.01 else '*' if p_kw < 0.05 else 'ns'
                color = _SIG_COL if sig != 'ns' else '#888'
                ax.text(0.5, 0.97, sig, transform=ax.transAxes, ha='center', va='top',
                        fontsize=14, fontweight='bold', color=color)
                ax.text(0.5, 0.91, f'p={p_kw:.3f}', transform=ax.transAxes,
                        ha='center', va='top', fontsize=8, color='#555')
            for i, (g, grp) in enumerate(zip(order, groups)):
                ax.text(i, ax.get_ylim()[0], f'n={len(grp)}',
                        ha='center', va='bottom', fontsize=8, color='#555')

            ax.set_title(label, fontsize=10, fontweight='bold', pad=8)
            ax.set_xlabel('')
            ax.set_xticklabels(order, fontsize=9)
            if ax == axes[0]:
                ax.set_ylabel('β (std units)', fontsize=10)
            sns.despine(ax=ax)

        fig.tight_layout()
        plt.show()

In [ ]:
# ── N significant targets per cell + predictor set comparison ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('How much does each predictor set explain?', fontsize=12)

# Left: histogram of n_sig per cell (waveform only)
axes[0].hist(df['n_sig_wv'], bins=range(0, len(target_names) + 2),
             color=_CB[0], alpha=0.85, edgecolor='white', linewidth=0.8)
axes[0].axvline(df['n_sig_wv'].mean(), color=_SIG_COL, lw=2, ls='--',
                label=f'Mean = {df["n_sig_wv"].mean():.1f}')
axes[0].set_xlabel('N FDR-significant targets per cell', fontsize=10)
axes[0].set_ylabel('N cells', fontsize=10)
axes[0].set_title('Waveform only — distribution across cells', fontsize=10)
axes[0].legend(fontsize=9, frameon=False)
axes[0].set_xticks(range(len(target_names) + 1))
sns.despine(ax=axes[0])

# Right: mean n_sig per predictor set with individual cell dots
pset_data = {
    'Waveform\nonly':      df['n_sig_wv'].values,
    'Log ISI\nonly':       df['n_sig_isi'].values,
    'Waveform\n+ Log ISI': df['n_sig_both'].values,
}
x = np.arange(len(pset_data))
rng = np.random.default_rng(42)
colors_bar = [_CB[0], _CB[1], _CB[2]]

for i, (lbl, vals) in enumerate(pset_data.items()):
    mean_v = np.mean(vals)
    sem_v  = stats.sem(vals)
    axes[1].bar(i, mean_v, color=colors_bar[i], alpha=0.6, width=0.5, edgecolor='white')
    axes[1].errorbar(i, mean_v, yerr=sem_v, fmt='none', color='k', capsize=5, lw=2)
    jitter = rng.uniform(-0.18, 0.18, len(vals))
    axes[1].scatter(i + jitter, vals, color=colors_bar[i], alpha=0.7, s=20, zorder=3)

# Pairwise Wilcoxon tests between waveform and ISI
_, p_wv_isi = stats.wilcoxon(df['n_sig_wv'], df['n_sig_isi'])
_, p_wv_bt  = stats.wilcoxon(df['n_sig_wv'], df['n_sig_both'])
for i, (j, p) in enumerate([(1, p_wv_isi), (2, p_wv_bt)]):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    y   = max(df['n_sig_wv'].max(), df['n_sig_isi'].max(), df['n_sig_both'].max()) + 0.5 + i * 0.8
    col = _SIG_COL if sig != 'ns' else '#888'
    axes[1].plot([0, j], [y, y], color=col, lw=1.2)
    axes[1].text((0 + j) / 2, y + 0.1, sig, ha='center', fontsize=11,
                 fontweight='bold', color=col)

axes[1].set_xticks(x)
axes[1].set_xticklabels(pset_data.keys(), fontsize=9)
axes[1].set_ylabel('Mean N significant targets per cell', fontsize=10)
axes[1].set_title('Predictor set comparison\n(Wilcoxon vs Waveform only)', fontsize=10)
sns.despine(ax=axes[1])

fig.tight_layout()
plt.show()